# READ FIRST: what this notebook uses and what you need to test it

## What it uses
An **ensemble** for every field, built in two layers (stacking):

| Layer | Models | Features |
|---|---|---|
| 1. base models | `lr` logistic regression, `svm` linear SVM, `nb` complement naive Bayes, `rf` **Random Forest**, `xgb` **XGBoost** (`lgbm` **LightGBM** optional) | linear models: the sparse TF-IDF (word 1-2 grams; the SVM also character 3-5 grams). Tree models: the text squeezed into 200 dense dimensions (truncated SVD of the TF-IDF) plus the context columns as numbers |
| 2. combiner | `stack`: a logistic regression that reads the **out-of-fold** scores of every base model. `vote`: the plain average of the base models' probabilities | the base models' predictions |

For every field the notebook picks one answer (`COMBINER = "auto"`): the candidate (a single base model, `vote` or `stack`) with the best **cross-validated score on the training tickets**. The test tickets are never used to choose, so the test scores stay honest. Set `COMBINER = "stack"` to always use the stacker.

| Field | Features (same as the hybrid) |
|---|---|
| `cmdb_ci`, `category`, `subcategory` | text only |
| `assignment_group` | text + contact type, hour, weekday, caller, created by |
| `priority` | text + impact |
| urgent (P1/P2) alert | P(P1) + P(P2) from the chosen priority model, threshold chosen on the training tickets' out-of-fold scores (catch 80% of real P1/P2) |
| `ci_mismatch` | the chosen CI model's probability compared with the CI the requester picked |

Trained on the **earliest 80%** of the export and scored on the **latest 20%**, exactly like `triage_hybrid_model_chart.ipynb`, so the two notebooks can be compared number for number.

## Files needed in this folder
- `raw_incident_reassign.csv`: the export of resolved/closed incidents (sections 3 to 7).
- `triage_ensemble_model.joblib` (`triage_ensemble_model_merged.joblib` when `MERGE_L1_INTO_ITSC = True`): the cached ensemble. Created by section 4 and loaded on later runs (set `RETRAIN_ENSEMBLE = True` after changing the settings).
- Packages: `scikit-learn`, `pandas`, `numpy`, `scipy`, `joblib`, `plotly`, `nbformat`, plus **`xgboost`** (and `lightgbm` only if you add `"lgbm"` to `BASE_MODELS`). A missing package only drops that model, with a message: `pip install xgboost lightgbm`.

## Columns needed to test new tickets
The same as the hybrid notebook: `short_description` and `description` are needed; `opened_at` (`dd-mm-yyyy hh:mm:ss`), `contact_type`, `caller_id`, `sys_created_by`, `impact` and `cmdb_ci` are optional (missing becomes `unknown`). See section 10.

## How to test
1. Run sections 0, 1 and 2 (imports and code).
2. Run section 4 (loads or builds the ensemble; the latest 20% is never used for training). **The first build is slow** (it trains every base model several times for the out-of-fold scores, see the timing chart in section 5); later runs load the file in seconds.
3. Section 5 shows which model wins each field, sections 6 and 7 the details, section 8 tests your own tickets.

# Ensemble triage model: Random Forest, XGBoost, linear models and a stacker

**Why stacking and not just one big Random Forest / XGBoost on the text?** Ticket text is tens of thousands of sparse TF-IDF columns. Linear models (logistic regression, SVM) are very good at that; tree models are slow on it and usually a few points worse. Trees are good at what linear models miss: **interactions** (this caller *and* this hour *and* this kind of text means that team). So each model gets the features it is good with, and a small logistic regression on top learns how much to trust each one, per class.

**How the stacker is trained without cheating:** the training tickets are split into `ENSEMBLE_FOLDS` parts. Each base model is trained on all parts but one and predicts the part it did not see, so every training ticket gets a prediction from a model that never saw it (out-of-fold). The stacker learns from those. Then every base model is trained once more on all training tickets, for use on new tickets.

| Section | What |
|---|---|
| 0, 1 | Imports, charts, the shared data code (copied from the hybrid notebook) |
| 2 | The CI text cleaning, the base models, the stacker and `ensemble_predict_df()` |
| 3 | The CIs the model is trained on |
| 4 | Build the ensemble on the earliest 80% (cached) |
| 5 | **Which model wins each field**: cross-validated on training, then on the latest 20%, and training time |
| 6 | Accuracy in detail on the latest 20% |
| 7 | Confusion matrix |
| 8 | Test your own tickets |
| 9 | (Optional) train the ensemble on all tickets and save it |
| 10 | Reference: CIs and columns |

## 0. Imports

In [ ]:
import string
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.svm import LinearSVC

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

warnings.filterwarnings("ignore", message="The least populated class")   # a class with fewer tickets than folds is fine: it is predicted from the folds that have it
warnings.filterwarnings("ignore", category=ConvergenceWarning)            # the logistic regression stops at max_iter=300, exactly like the hybrid's
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 120)

## Charts (plotly)

The cells that show data draw interactive plotly charts (hover for the exact numbers, click a legend entry to hide a series). Nothing here trains anything: the charts read the cached ensemble (`triage_ensemble_model.joblib`) and the CSV. The renderer below works in VS Code, in Jupyter and in a browser, offline; the charts are stored in the notebook file, so the first chart adds a few MB (the plotly.js library). Change `PLOTLY_RENDERER` to `"browser"` to open every chart in a browser tab instead.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

PLOTLY_RENDERER = "plotly_mimetype+notebook"     # VS Code, Jupyter and a browser, offline. "browser" opens each chart in a browser tab
pio.renderers.default = PLOTLY_RENDERER
INDIGO, GREY, LIGHT, GREEN, RED, AMBER = "#4f46e5", "#94a3b8", "#cbd5e1", "#16a34a", "#dc2626", "#f59e0b"


def show(fig, height=None, width=None):
    fig.update_layout(template="plotly_white", font=dict(size=13), margin=dict(l=70, r=40, t=90, b=70), legend=dict(orientation="h", y=-0.18))
    if height:
        fig.update_layout(height=height)
    if width:
        fig.update_layout(width=width)
    fig.show()


def pct(v) -> str:
    return "-" if v is None or v != v else f"{v:.1%}"


def table_figure(df: pd.DataFrame, title: str, col_widths=None, fill_colors=None, height=None):
    # A plotly table of a DataFrame (its index becomes the first column).
    d = df.reset_index()
    fig = go.Figure(go.Table(columnwidth=col_widths, header=dict(values=[f"<b>{c}</b>" for c in d.columns], fill_color="#e0e7ff", align="left"),
                             cells=dict(values=[d[c].astype(str).tolist() for c in d.columns], align="left", fill_color=fill_colors, height=28)))
    fig.update_layout(title=title)
    show(fig, height or 120 + 30 * (len(d) + 1))

# One fixed color per model, the same in every chart (a model keeps its color whatever is shown)
MODEL_COLOR = {"lr": "#2a78d6", "svm": "#eb6834", "nb": "#1baf7a", "rf": "#eda100", "xgb": "#e87ba4", "lgbm": "#008300", "vote": "#6250d6", "stack": "#e34948"}
MODEL_NAME = {"lr": "logistic regression", "svm": "linear SVM", "nb": "naive Bayes", "rf": "Random Forest", "xgb": "XGBoost", "lgbm": "LightGBM",
              "vote": "vote (average)", "stack": "stack (LR on top)"}


## 1. The shared data code

Copied from the hybrid notebook: the settings, the data preparation, the time split, the TF-IDF / one-hot features (`design()`) and the text-only urgency gate (`p_urgent_text`, recorded only). The per-field models are now in section 2.

In [ ]:
MERGE_L1_INTO_ITSC = True   # treat IMG-SMC-TCC-L1-Engineers as part of IMG-SMC-TCC-ITSC (same team); False = keep them separate
PRIORITY_CLASS_WEIGHT = None   # None or "balanced". balanced finds P2 more often (11% to 39% on the test tickets) but priority accuracy falls about 3 points, and the urgent alert is no better
TAG = ("_merged" if MERGE_L1_INTO_ITSC else "") + ("_balanced" if PRIORITY_CLASS_WEIGHT else "")   # keeps the model files of each setting apart
CSV = Path("raw_incident_reassign.csv")
URGENT = ["P1", "P2"]
DEFAULT_GROUP = "IMG-SMC-TCC-ITSC"
AUTO_ROUTE_CONF = 0.9        # assignment_group confidence for auto-routing
CI_MISMATCH_CONF = 0.8       # only flag a submitted-CI mismatch when the text prediction is this confident
GATE_TARGET_RECALL = 0.5     # urgency gate: share of real P1/P2 tickets it should catch (PROD's deployed value). The gate is recorded (p_urgent_text) but is no longer the alert
URGENT_TARGET_RECALL = 0.8   # urgent alert = P(P1)+P(P2) from the priority model; its threshold catches this share of the real P1/P2 tickets (chosen on the training tickets)
CTX, CALLER = ["contact_type", "hour", "weekday"], ["caller_id", "sys_created_by"]
INPUT_COLUMNS = ["short_description", "description", "opened_at", "contact_type", "caller_id", "sys_created_by", "impact", "cmdb_ci"]


def count_from_group(group: str) -> int:
    return 1 if group == DEFAULT_GROUP else 2 if group.startswith("IMG-SMC-TCC-") else 3


def choose_threshold(is_urgent, urgent_probability, target_recall: float) -> float:
    # Highest probability threshold that still reaches the recall target (same rule as PROD's priority_classifier).
    _, recalls, thresholds = precision_recall_curve(is_urgent, urgent_probability)
    reachable = [t for t, r in zip(thresholds, recalls[: len(thresholds)]) if r >= target_recall]
    return float(max(reachable)) if reachable else float(np.min(thresholds))


def prepare(df: pd.DataFrame) -> pd.DataFrame:
    # Same preparation for training rows and live tickets: missing columns / values become "unknown".
    df = df.reindex(columns=[*df.columns, *[c for c in INPUT_COLUMNS if c not in df.columns]]).copy()
    df["text"] = df["short_description"].fillna("") + " " + df["description"].fillna("")
    opened = pd.to_datetime(df["opened_at"], dayfirst=True, errors="coerce")
    df["hour"] = opened.dt.hour.fillna(-1).astype(int).astype(str)
    df["weekday"] = opened.dt.weekday.fillna(-1).astype(int).astype(str)
    for c in ("contact_type", "caller_id", "sys_created_by", "impact"):
        df[c] = df[c].fillna("unknown")
    return df


def load_data() -> pd.DataFrame:
    df = pd.read_csv(CSV, low_memory=False).dropna(subset=["subcategory"])
    df["assignment_group"] = df["assignment_group"].str.strip()
    if MERGE_L1_INTO_ITSC:
        df["assignment_group"] = df["assignment_group"].replace("IMG-SMC-TCC-L1-Engineers", DEFAULT_GROUP)
    df["priority_label"] = "P" + df["priority"].str[0]
    return prepare(df)


def split_time(df: pd.DataFrame):
    # Earliest 80% of tickets to train, latest 20% to test: closer to real use than a random split.
    ordered = df.assign(_t=pd.to_datetime(df["opened_at"], dayfirst=True, errors="coerce")).sort_values("_t", na_position="first")
    cut = int(len(ordered) * 0.8)
    return ordered.iloc[:cut].drop(columns="_t"), ordered.iloc[cut:].drop(columns="_t")


def design(d: pd.DataFrame, vec, enc: dict) -> dict:
    # Feature matrices per model: text only | text + context + caller (group) | text + impact (priority).
    text = vec.transform(d["text"])
    return {"text": text,
            "group": hstack([text, enc["ctx"].transform(d[CTX]), enc["caller"].transform(d[CALLER])]).tocsr(),
            "priority": hstack([text, enc["impact"].transform(d[["impact"]])]).tocsr()}


def _fit_gate(X, urgent):
    gate = LogisticRegression(max_iter=300, C=10, class_weight="balanced")
    oof = cross_val_predict(gate, X, urgent, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
    return gate.fit(X, urgent), choose_threshold(urgent, oof, GATE_TARGET_RECALL)


def fit_shared(train: pd.DataFrame) -> dict:
    # The word TF-IDF and the one-hot encoders (the linear models' features) and the text-only urgency gate.
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2).fit(train["text"])
    enc = {"ctx": OneHotEncoder(handle_unknown="infrequent_if_exist").fit(train[CTX]),
           "caller": OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5).fit(train[CALLER]),   # rare callers share one bucket
           "impact": OneHotEncoder(handle_unknown="ignore").fit(train[["impact"]])}
    gate, gate_threshold = _fit_gate(vec.transform(train["text"]), np.isin(train["priority_label"], URGENT).astype(int))
    return {"vectorizer": vec, "encoders": enc, "gate": gate, "gate_threshold": gate_threshold}


## 2. The base models, the stacker and the predict functions

| Setting | Meaning |
|---|---|
| `BASE_MODELS` | the base models to train. Remove one to skip it (`"xgb"` is the slowest on fields with many classes); add `"lgbm"` for LightGBM |
| `ENSEMBLE_FOLDS` | out-of-fold parts for the stacker. `3` is a good default; `5` is a bit better and about 1.5 times slower |
| `COMBINER` | `"auto"`: per field, the candidate with the best cross-validated training score (`CHOOSE_BY`). `"stack"` or `"vote"`: always that one. A base model name (`"lr"`, `"rf"`, ...) uses that model alone |
| `CHOOSE_BY` | `"accuracy"` or `"macro_f1"` (macro F1 gives every class the same weight, so it favours the small CIs and teams) |
| `SVD_COMPONENTS` | dense text dimensions for the tree models |
| `RF_TREES`, `XGB_ROUNDS` | size of the Random Forest and of XGBoost / LightGBM. XGBoost builds one tree per class per round, so it is the slowest on fields with many classes; its settings (depth 4, 64 bins) are chosen to keep it about 7 times faster than the defaults for the same accuracy |

The SVM gives scores, not probabilities, so it is used by the stacker but not by `vote`, and it cannot be the chosen answer on its own (`auto_route`, `ci_mismatch` and the urgent alert need a probability). The stacker turns every base model's output, the SVM's included, into a probability.

In [ ]:
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS)
PUNCTUATION = str.maketrans({c: " " for c in string.punctuation})   # punctuation becomes a space: "DA-VisionAir" is "da visionair", not "davisionair"
CI_CLEANING = "punct_space"   # stored with the hybrid: a model trained with another cleaning must not be used with this code (the old one deleted punctuation)


def clean_text(text) -> str:
    # lower case, punctuation turned into spaces, digits turned into spaces (asset tags are noise), no English stop words
    t = str(text).lower().translate(PUNCTUATION)
    t = "".join(" " if ch.isdigit() else ch for ch in t)
    return " ".join(w for w in t.split() if w not in STOP_WORDS)


def ci_fit_features(d: pd.DataFrame) -> dict:
    cleaned = d["text"].map(clean_text)
    return {"word": TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.9, max_features=20_000, sublinear_tf=True).fit(cleaned),
            "char": TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.9, max_features=30_000, sublinear_tf=True).fit(cleaned)}


def ci_design(d: pd.DataFrame, features: dict):
    cleaned = d["text"].map(clean_text)
    return hstack([features["word"].transform(cleaned), features["char"].transform(cleaned)]).tocsr()


BASE_MODELS = ["lr", "svm", "nb", "rf", "xgb"]     # add "lgbm" for LightGBM
ENSEMBLE_FOLDS = 3
COMBINER = "auto"           # "auto" | "stack" | "vote" | a base model name
CHOOSE_BY = "accuracy"      # "accuracy" | "macro_f1"
SVD_COMPONENTS = 200
RF_TREES = 200
XGB_ROUNDS = 100

FIELDS = ["cmdb_ci", "assignment_group", "category", "subcategory", "priority"]
TARGET_COL = {"cmdb_ci": "cmdb_ci", "assignment_group": "assignment_group", "category": "category", "subcategory": "subcategory", "priority": "priority_label"}
FEATURE_OF = {"cmdb_ci": "text", "assignment_group": "group", "category": "text", "subcategory": "text", "priority": "priority"}
FAMILY = {"lr": "linear", "nb": "linear", "svm": "svm", "rf": "tree", "xgb": "tree", "lgbm": "tree", "meta": "meta"}
PROBA_MODELS = ["lr", "nb", "rf", "xgb", "lgbm"]          # give probabilities (the SVM gives scores)
PARALLEL_FOLDS = {"lr", "svm", "nb", "meta"}              # single-threaded models: their folds run side by side. Trees use every core themselves
TREE_CATEGORICAL = ["contact_type", "caller_id", "sys_created_by", "impact"]
TREE_CONTEXT = {"text": [], "group": ["contact_type", "hour", "weekday", "caller_id", "sys_created_by"], "priority": ["impact"]}
MISSING_PACKAGE = {"xgb": XGBClassifier is None, "lgbm": LGBMClassifier is None}


def available_models(models) -> list:
    missing = [m for m in models if MISSING_PACKAGE.get(m)]
    if missing:
        print(f"skipping {missing}: package not installed (pip install xgboost lightgbm)")
    return [m for m in models if m not in missing]


def make_model(kind: str, field: str, n_classes: int):
    if kind == "lr":
        return LogisticRegression(max_iter=300, C=10, class_weight=PRIORITY_CLASS_WEIGHT if field == "priority" else None)
    if kind == "svm":
        return LinearSVC(C=0.5, class_weight="balanced" if field == "cmdb_ci" else None, random_state=42, max_iter=5000)     # the hybrid's tuned CI setting
    if kind == "nb":
        return ComplementNB(alpha=0.3)
    if kind == "rf":
        return RandomForestClassifier(n_estimators=RF_TREES, max_features="sqrt", min_samples_leaf=2, n_jobs=-1, random_state=42)
    if kind == "xgb":
        return XGBClassifier(n_estimators=XGB_ROUNDS, max_depth=4, learning_rate=0.2, subsample=0.8, colsample_bytree=0.5, tree_method="hist", max_bin=64, n_jobs=-1, random_state=42)
    if kind == "lgbm":
        return LGBMClassifier(n_estimators=XGB_ROUNDS, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.5, max_bin=63, n_jobs=-1, random_state=42, verbose=-1)
    if kind == "meta":     # the stacker: a logistic regression on the base models' probabilities
        return LogisticRegression(max_iter=1000, C=1.0)
    raise ValueError(kind)


# ---- features: fitted once on the training tickets
def fit_ensemble_features(train: pd.DataFrame, shared: dict) -> dict:
    t0 = time.time()
    features = {"svm_text": ci_fit_features(train),        # cleaned word 1-2 grams + character 3-5 grams (the hybrid's CI features)
                "svd": TruncatedSVD(SVD_COMPONENTS, random_state=42).fit(shared["vectorizer"].transform(train["text"])),
                "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, min_frequency=5).fit(train[TREE_CATEGORICAL])}
    print(f"  features fitted in {time.time() - t0:.0f}s")
    return features


def tree_context(d: pd.DataFrame, features: dict) -> pd.DataFrame:
    # The context as numbers for the trees: hour and weekday as they are (-1 = unknown), the other columns as category codes.
    ctx = pd.DataFrame(features["ordinal"].transform(d[TREE_CATEGORICAL]), columns=TREE_CATEGORICAL, index=d.index)
    ctx["hour"], ctx["weekday"] = d["hour"].astype(int), d["weekday"].astype(int)
    return ctx


def ensemble_design(d: pd.DataFrame, features: dict, shared: dict) -> dict:
    # Feature matrices per family (linear | svm | tree) and per feature set (text | group | priority).
    linear = design(d, shared["vectorizer"], shared["encoders"])
    enc = shared["encoders"]
    svm_text = ci_design(d, features["svm_text"])
    svm = {"text": svm_text,
           "group": hstack([svm_text, enc["ctx"].transform(d[CTX]), enc["caller"].transform(d[CALLER])]).tocsr(),
           "priority": hstack([svm_text, enc["impact"].transform(d[["impact"]])]).tocsr()}
    svd, ctx = features["svd"].transform(linear["text"]), tree_context(d, features)
    tree = {k: np.hstack([svd, ctx[cols].to_numpy(float)]) if cols else svd for k, cols in TREE_CONTEXT.items()}
    return {"linear": linear, "svm": svm, "tree": tree}


# ---- one model, any subset of the classes: the scores always come back with one column per class of the field
def fit_one(kind: str, field: str, X, y: np.ndarray, n_classes: int):
    present = np.unique(y)
    if len(present) == 1:
        return None, present
    return make_model(kind, field, n_classes).fit(X, np.searchsorted(present, y)), present


def scores_of(kind: str, fitted, X, n_classes: int) -> np.ndarray:
    model, present = fitted
    out = np.zeros((X.shape[0], n_classes))
    if model is None:
        out[:, present[0]] = 1.0
        return out
    if kind == "svm":
        s = model.decision_function(X)
        s = np.column_stack([-s, s]) if s.ndim == 1 else s
        out[:] = s.min() - 1.0           # a class the model never saw gets the lowest score
        out[:, present] = s
    else:
        out[:, present] = model.predict_proba(X)
    return out


def out_of_fold(kind: str, field: str, X, y: np.ndarray, n_classes: int, folds: int) -> np.ndarray:
    # Every training ticket scored by a model that did not see it.
    splits = list(StratifiedKFold(folds, shuffle=True, random_state=42).split(np.zeros(len(y)), y))

    def one(tr, te):
        return te, scores_of(kind, fit_one(kind, field, X[tr], y[tr], n_classes), X[te], n_classes)

    if kind in PARALLEL_FOLDS:
        parts = Parallel(n_jobs=folds)(delayed(one)(tr, te) for tr, te in splits)
    else:
        parts = [one(tr, te) for tr, te in splits]
    out = np.zeros((len(y), n_classes))
    for te, s in parts:
        out[te] = s
    return out


def softmax(s: np.ndarray) -> np.ndarray:
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


SVM_SHARPNESS = 3.0     # the SVM's scores (about -1.5 to 1) through a softmax, made sharper: probability-like inputs for the stacker


def as_proba(m: str, s: np.ndarray) -> np.ndarray:
    return softmax(SVM_SHARPNESS * s) if m == "svm" else s


def meta_features(base: dict, models: list) -> np.ndarray:
    # The stacker reads every base model's probabilities side by side (unscaled: on raw scaled scores it overfits the many small classes).
    return np.hstack([as_proba(m, base[m]) for m in models])


def candidates(base: dict, models: list, meta_proba: np.ndarray) -> dict:
    # Every possible answer: each base model alone, the average of the probability models, and the stacker.
    out = {m: as_proba(m, base[m]) for m in models}
    proba_models = [m for m in models if m in PROBA_MODELS]
    if len(proba_models) >= 2:
        out["vote"] = np.mean([base[m] for m in proba_models], axis=0)
    out["stack"] = meta_proba
    return out


def score(y: np.ndarray, proba: np.ndarray) -> dict:
    pred = proba.argmax(1)
    return {"accuracy": float((pred == y).mean()), "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0))}


def fit_ensemble(train: pd.DataFrame, models: list, folds: int) -> dict:
    t_start = time.time()
    models = available_models(models)
    shared = fit_shared(train)
    features = fit_ensemble_features(train, shared)
    M = ensemble_design(train, features, shared)
    fields, oof_rows, time_rows = {}, [], []
    for field in FIELDS:
        y_raw = train[TARGET_COL[field]].astype(str).to_numpy()
        classes = np.unique(y_raw)
        y, K = np.searchsorted(classes, y_raw), len(classes)
        oof, fitted = {}, {}
        for m in models:
            t0 = time.time()
            X = M[FAMILY[m]][FEATURE_OF[field]]
            oof[m] = out_of_fold(m, field, X, y, K, folds)
            fitted[m] = fit_one(m, field, X, y, K)
            time_rows.append({"field": field, "model": m, "seconds": time.time() - t0})
            print(f"  {field:<17} {m:<5} {time.time() - t0:6.0f}s   out-of-fold accuracy {score(y, oof[m])['accuracy']:.1%}")
        t0 = time.time()
        meta_X = meta_features(oof, models)
        meta_oof = out_of_fold("meta", field, meta_X, y, K, folds)       # the stacker's own honest score: a second round of folds
        meta = fit_one("meta", field, meta_X, y, K)
        time_rows.append({"field": field, "model": "stack", "seconds": time.time() - t0})
        cand = candidates(oof, models, meta_oof)
        for name, proba in cand.items():
            oof_rows.append({"field": field, "model": name, **score(y, proba)})
        allowed = {k: v for k, v in cand.items() if k != "svm"}
        if COMBINER == "auto":
            chosen = max(allowed, key=lambda k: score(y, allowed[k])[CHOOSE_BY])
        else:
            assert COMBINER in allowed, f"COMBINER must be 'auto' or one of {list(allowed)}"
            chosen = COMBINER
        print(f"  {field:<17} -> {chosen} (out-of-fold {CHOOSE_BY} {score(y, cand[chosen])[CHOOSE_BY]:.1%})")
        fields[field] = {"classes": classes, "models": fitted, "meta": meta, "chosen": chosen}
        if field == "priority":       # the urgent threshold comes from the chosen priority model's out-of-fold probabilities, never from test tickets
            urgent_cols = [i for i, c in enumerate(classes) if c in URGENT]
            urgent_threshold = choose_threshold(np.isin(y_raw, URGENT).astype(int), cand[chosen][:, urgent_cols].sum(1), URGENT_TARGET_RECALL)
    print(f"ensemble trained in {(time.time() - t_start) / 60:.1f} minutes")
    return {"shared": shared, "features": features, "fields": fields, "models": models, "urgent_threshold": urgent_threshold,
            "oof_scores": pd.DataFrame(oof_rows), "train_seconds": pd.DataFrame(time_rows),
            "settings": {"base_models": models, "folds": folds, "combiner": COMBINER, "choose_by": CHOOSE_BY, "svd_components": SVD_COMPONENTS,
                         "rf_trees": RF_TREES, "xgb_rounds": XGB_ROUNDS, "ci_cleaning": CI_CLEANING}}


# ---- predicting
def field_candidates(M: dict, ens: dict, field: str) -> dict:
    f, models = ens["fields"][field], ens["models"]
    K = len(f["classes"])
    base = {m: scores_of(m, f["models"][m], M[FAMILY[m]][FEATURE_OF[field]], K) for m in models}
    return candidates(base, models, scores_of("meta", f["meta"], meta_features(base, models), K))


def ensemble_candidates_df(df: pd.DataFrame, ens: dict) -> dict:
    # Every candidate's predicted label for every field (to compare them on test tickets): {field: DataFrame with one column per candidate}.
    M = ensemble_design(prepare(df), ens["features"], ens["shared"])
    out = {}
    for field in FIELDS:
        classes = ens["fields"][field]["classes"]
        out[field] = pd.DataFrame({name: classes[p.argmax(1)] for name, p in field_candidates(M, ens, field).items()}, index=df.index)
    return out


def ensemble_predict_df(df: pd.DataFrame, ens: dict, submitted_ci=None) -> pd.DataFrame:
    # Same output columns as the hybrid's hybrid_predict_df(), from the chosen candidate of every field.
    assert ens.get("settings", {}).get("ci_cleaning") == CI_CLEANING, "this ensemble was trained with another text cleaning: set RETRAIN_ENSEMBLE = True in section 4"
    d = prepare(df)
    if submitted_ci is None:
        submitted_ci = d["cmdb_ci"]
    M = ensemble_design(d, ens["features"], ens["shared"])
    out = pd.DataFrame(index=d.index)
    for field in FIELDS:
        f = ens["fields"][field]
        proba = field_candidates(M, ens, field)[f["chosen"]]
        out[field], out[f"{field}_conf"] = f["classes"][proba.argmax(1)], proba.max(1)
        if field == "priority":
            out["p_urgent_priority"] = proba[:, [i for i, c in enumerate(f["classes"]) if c in URGENT]].sum(1)
    out["auto_route"] = out["assignment_group_conf"] >= AUTO_ROUTE_CONF
    out["reassignment_count"] = out["assignment_group"].map(count_from_group)
    out["p_urgent_text"] = ens["shared"]["gate"].predict_proba(M["linear"]["text"])[:, 1]      # recorded only
    out["urgent_alert"] = (out["p_urgent_priority"] >= ens["urgent_threshold"]).to_numpy()
    out["alert_reason"] = np.where(out["urgent_alert"], "priority score", "")
    out["ci_mismatch"] = (submitted_ci.fillna("").astype(str).str.strip().ne("") & (submitted_ci != out["cmdb_ci"]) & (out["cmdb_ci_conf"] >= CI_MISMATCH_CONF)).to_numpy()
    return out


def ensemble_predict_ticket(ticket: dict, ens: dict) -> dict:
    return ensemble_predict_df(pd.DataFrame([ticket]), ens).iloc[0].to_dict()


## 3. The CIs the model is trained on

The CIs the model can predict, with how many tickets of each are in the training part (earliest 80%) and in the test part (latest 20%). Needs the imports and the code cells and the CSV; it does not train anything.

In [ ]:
ci_all = load_data()
ci_train, ci_test = split_time(ci_all)
trained_cis = (ci_train["cmdb_ci"].value_counts().rename("tickets in training").to_frame()
               .join(ci_test["cmdb_ci"].value_counts().rename("tickets in test"), how="left").fillna(0).astype(int))
trained_cis["share of training"] = trained_cis["tickets in training"] / trained_cis["tickets in training"].sum()

fig = go.Figure()
fig.add_bar(y=trained_cis.index, x=trained_cis["tickets in training"], orientation="h", name="tickets in training (earliest 80%)", marker_color=INDIGO,
            text=trained_cis["tickets in training"], textposition="outside", customdata=trained_cis["share of training"],
            hovertemplate="%{y}<br>%{x} training tickets (%{customdata:.1%} of training)<extra></extra>")
fig.add_bar(y=trained_cis.index, x=trained_cis["tickets in test"], orientation="h", name="tickets in test (latest 20%)", marker_color=GREY,
            text=trained_cis["tickets in test"], textposition="outside", hovertemplate="%{y}<br>%{x} test tickets<extra></extra>")
fig.update_xaxes(type="log", title="tickets (log scale; a CI with 0 test tickets has no grey bar)")
fig.update_yaxes(autorange="reversed")
fig.update_layout(barmode="group", title=f"The model is trained on {len(trained_cis)} CIs ({len(ci_train)} training tickets, {len(ci_test)} test tickets)")
show(fig, height=560)

## 4. Build the ensemble on the earliest 80% of tickets

Trains every base model `ENSEMBLE_FOLDS + 1` times per field (the out-of-fold rounds plus the final fit) and the stacker, then caches everything in `triage_ensemble_model.joblib`. It prints each model's time and out-of-fold accuracy as it goes. **Expect about half an hour on 38k tickets on a 4-core machine** (several times the hybrid's 4 minutes), most of it in XGBoost and the Random Forest on the fields with many classes (`assignment_group`, `subcategory`); remove `"xgb"` from `BASE_MODELS` or use `ENSEMBLE_FOLDS = 2` for a quicker first look. Later runs load the file in seconds. The latest 20% are never used for training or for choosing.

If you change any setting of section 2, set `RETRAIN_ENSEMBLE = True` once.

In [ ]:
ENSEMBLE_PATH = Path(f"triage_ensemble_model{TAG}.joblib")      # the ensemble, earliest 80%
RETRAIN_ENSEMBLE = False

all_tickets = load_data()
split_train, split_test = split_time(all_tickets)
when = pd.to_datetime(split_test["opened_at"], dayfirst=True)
print(f"{len(all_tickets)} tickets: {len(split_train)} train (earliest) / {len(split_test)} test (latest, {when.min():%Y-%m-%d} to {when.max():%Y-%m-%d})")

if ENSEMBLE_PATH.exists() and not RETRAIN_ENSEMBLE:
    ens_model = joblib.load(ENSEMBLE_PATH)             # a pickle: only load a file you trained yourself
    print(f"loaded the cached ensemble {ENSEMBLE_PATH.name} ({ENSEMBLE_PATH.stat().st_size / 1e6:.0f} MB), settings: {ens_model['settings']}")
else:
    print(f"training the ensemble {BASE_MODELS} with {ENSEMBLE_FOLDS} folds on the earliest {len(split_train)} tickets ...")
    ens_model = fit_ensemble(split_train, BASE_MODELS, ENSEMBLE_FOLDS)
    joblib.dump(ens_model, ENSEMBLE_PATH, compress=3)
    print(f"ensemble cached as {ENSEMBLE_PATH.name} ({ENSEMBLE_PATH.stat().st_size / 1e6:.0f} MB)")
print("chosen per field:", {f: ens_model["fields"][f]["chosen"] for f in FIELDS})


## 5. Which model wins each field?

1. **Cross-validated on the training tickets** (out-of-fold): this is what `COMBINER = "auto"` chose from (marked ★). Choosing on the test tickets instead would make the test scores look better than they are.
2. **On the latest 20%** (never seen): every candidate on the same test tickets, so you can check that the choice holds up on newer tickets.
3. **Macro F1** on the latest 20%: does the model find the small classes too?
4. **CI recall per class**: the stacker against the logistic regression and the SVM alone (the SVM is close to the hybrid's CI model).
5. **Training time** per base model, to decide which ones are worth keeping.

The tables under the heatmaps hold the exact numbers.

In [ ]:
assert "ens_model" in globals(), "Run section 4 first"
cand_test = ensemble_candidates_df(split_test, ens_model)
chosen = {f: ens_model["fields"][f]["chosen"] for f in FIELDS}
order = [m for m in [*ens_model["models"], "vote", "stack"] if m in cand_test[FIELDS[0]].columns]


def score_heatmap(table: pd.DataFrame, title: str, colorbar_title: str):
    # Fields (rows) x models (columns), the chosen model of each field marked with a star.
    z = table.loc[FIELDS, order]
    text = [[f"{'★ ' if chosen[f] == m else ''}{z.loc[f, m]:.1%}" for m in order] for f in FIELDS]
    fig = go.Figure(go.Heatmap(z=z.values, x=[MODEL_NAME[m] for m in order], y=FIELDS, text=text, texttemplate="%{text}", colorscale="Blues",
                               colorbar=dict(title=colorbar_title, tickformat=".0%"), hovertemplate="%{y}<br>%{x}: %{z:.2%}<extra></extra>"))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=title, margin=dict(l=170))
    show(fig, height=420)


# 1. cross-validated on training
oof = ens_model["oof_scores"]
score_heatmap(oof.pivot(index="field", columns="model", values="accuracy"),
              f"1. Out-of-fold accuracy on the {len(split_train)} training tickets ({ens_model['settings']['folds']} folds)<br><sup>★ = chosen for the field (by {ens_model['settings']['choose_by']}); "
              "the SVM alone is shown but never chosen (it gives scores, not probabilities)</sup>", "accuracy")
show_table = oof.pivot(index="field", columns="model", values=ens_model["settings"]["choose_by"]).loc[FIELDS, order].apply(lambda col: col.map(pct))
show_table.columns = [MODEL_NAME[m] for m in order]
table_figure(show_table, f"Out-of-fold {ens_model['settings']['choose_by']} (training tickets), the numbers the choice was made on", height=420)

# 2. and 3. on the latest 20%
rows = []
for f in FIELDS:
    y = split_test[TARGET_COL[f]].astype(str)
    for m in order:
        rows.append({"field": f, "model": m, "accuracy": (cand_test[f][m] == y).mean(), "macro_f1": f1_score(y, cand_test[f][m], average="macro", zero_division=0)})
test_scores = pd.DataFrame(rows)
baseline = {f: (split_test[TARGET_COL[f]] == split_train[TARGET_COL[f]].mode()[0]).mean() for f in FIELDS}
for metric, label, number in (("accuracy", "accuracy", "2"), ("macro_f1", "macro F1 (every class counts the same)", "3")):
    fig = go.Figure()
    if metric == "accuracy":
        fig.add_bar(x=FIELDS, y=[baseline[f] for f in FIELDS], name="always the most common value", marker_color=LIGHT, hovertemplate="%{x}<br>baseline %{y:.1%}<extra></extra>")
    for m in order:
        d = test_scores[test_scores["model"] == m].set_index("field").loc[FIELDS]
        fig.add_bar(x=FIELDS, y=d[metric], name=MODEL_NAME[m], marker_color=MODEL_COLOR[m], marker_line=dict(color="white", width=2),
                    hovertemplate="%{x}<br>" + MODEL_NAME[m] + ": %{y:.2%}<extra></extra>")
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", title=label)
    fig.update_layout(barmode="group", bargap=0.2, title=f"{number}. {label.split(' (')[0].capitalize()} of every model on the latest {len(split_test)} tickets (never seen in training)<br><sup>hover for the exact numbers; the table below has them all</sup>")
    show(fig, height=500)
    t = test_scores.pivot(index="field", columns="model", values=metric).loc[FIELDS, order]
    marked = pd.DataFrame({MODEL_NAME[m]: [("★ " if chosen[f] == m else "") + pct(t.loc[f, m]) for f in FIELDS] for m in order}, index=pd.Index(FIELDS, name="field"))
    if metric == "accuracy":
        marked.insert(0, "baseline", [pct(baseline[f]) for f in FIELDS])
    table_figure(marked, f"{label.split(' (')[0].capitalize()} on the latest 20% (★ = the model chosen on the training tickets)", height=420)

# 4. CI recall per class
train_n = split_train["cmdb_ci"].value_counts()
actual = split_test["cmdb_ci"].astype(str)
ci_models = [m for m in ("lr", "svm", chosen["cmdb_ci"]) if m in order]
ci_models = list(dict.fromkeys(ci_models))
rec = pd.DataFrame([{"CI": cls, "training tickets": int(train_n.get(cls, 0)), "test tickets": int(n),
                     **{m: (cand_test["cmdb_ci"][m][actual == cls] == cls).mean() for m in ci_models}} for cls, n in actual.value_counts().items()]).sort_values("training tickets")
ylabels = [f"{c}  ({a} training, {b} test)" for c, a, b in zip(rec["CI"], rec["training tickets"], rec["test tickets"])]
fig = go.Figure()
for m in ci_models:
    fig.add_bar(y=ylabels, x=rec[m], orientation="h", name=MODEL_NAME[m] + (" (chosen)" if m == chosen["cmdb_ci"] else ""), marker_color=MODEL_COLOR[m],
                text=[f"{v:.0%}" for v in rec[m]] if m == chosen["cmdb_ci"] else None, textposition="outside", hovertemplate="%{y}<br>recall %{x:.1%}<extra></extra>")
fig.update_xaxes(range=[0, 1.12], tickformat=".0%", title="recall (share of the CI's test tickets found)")
fig.update_yaxes(autorange="reversed")
fig.update_layout(barmode="group", title="4. CI recall per class (smallest class on top): the chosen model against the logistic regression and the SVM alone")
show(fig, height=560)

# 5. training time
secs = ens_model["train_seconds"]
fig = go.Figure()
for m in [*ens_model["models"], "stack"]:
    d = secs[secs["model"] == m].set_index("field").reindex(FIELDS)
    fig.add_bar(y=FIELDS, x=d["seconds"] / 60, orientation="h", name=MODEL_NAME[m], marker_color=MODEL_COLOR[m], marker_line=dict(color="white", width=2),
                hovertemplate="%{y}<br>" + MODEL_NAME[m] + ": %{x:.1f} min<extra></extra>")
fig.update_xaxes(title="minutes (out-of-fold rounds + final fit)")
fig.update_yaxes(autorange="reversed")
fig.update_layout(barmode="stack", legend=dict(traceorder="normal"), title=f"5. Training time per field and model: {secs['seconds'].sum() / 60:.0f} minutes in all (plus the shared features)")
show(fig, height=420)


## 6. Accuracy in detail on the latest 20%

The ensemble (the chosen model of every field) scored on the latest 20%: accuracy of every field against "always guess the most common value", the assignment group in detail, confidence against coverage, the urgent (P1/P2) tickets, and `split_results` with every test ticket (actual against predicted). 

In [ ]:
# Test on the latest 20%: the ensemble from section 4 scores tickets it has never seen. No training here.
assert "ens_model" in globals(), "Run section 4 (build the ensemble) first"
st_out = ensemble_predict_df(split_test, ens_model)
TRUTH_COL = {"cmdb_ci": "cmdb_ci", "assignment_group": "assignment_group", "category": "category", "subcategory": "subcategory",
             "priority": "priority_label", "reassignment_count": "reassignment_count"}
MODE = f"HELD-OUT: trained on the earliest {len(split_train)} tickets, scored on the latest {len(split_test)} ({when.min():%Y-%m-%d} to {when.max():%Y-%m-%d}), so the scores are honest"

# 1. accuracy against "always guess the most common value"
rows = [{"field": f, "accuracy": (st_out[f] == split_test[c]).mean(), "majority baseline": (split_test[c] == split_train[c].mode()[0]).mean()} for f, c in TRUTH_COL.items()]
st_summary = pd.DataFrame(rows).set_index("field")
st_summary["gain"] = st_summary["accuracy"] - st_summary["majority baseline"]
fig = go.Figure()
fig.add_bar(x=st_summary.index, y=st_summary["majority baseline"], name="always the most common value", marker_color=LIGHT, text=[pct(v) for v in st_summary["majority baseline"]], textposition="outside")
fig.add_bar(x=st_summary.index, y=st_summary["accuracy"], name="ensemble", marker_color=INDIGO, text=[f"{pct(a)}<br>({g * 100:+.1f} pts)" for a, g in zip(st_summary["accuracy"], st_summary["gain"])], textposition="outside")
fig.update_yaxes(range=[0, 1.18], tickformat=".0%", title="accuracy")
fig.update_layout(barmode="group", title="1. Accuracy against always guessing the most common value<br><sup>" + MODE + "</sup>")
show(fig, height=500)

# 2. assignment_group in detail
y = split_test["assignment_group"]
hit = st_out["assignment_group"] == y
is_itsc = y == DEFAULT_GROUP
detail = pd.Series({"specialist-team recall": hit[~is_itsc].mean(), "ITSC recall": hit[is_itsc].mean(),
                    f"auto-route: share of tickets (confidence >= {AUTO_ROUTE_CONF})": st_out["auto_route"].mean(), "auto-route: share of those that are correct": hit[st_out["auto_route"]].mean()})
fig = go.Figure(go.Bar(y=detail.index, x=detail.values, orientation="h", marker_color=[INDIGO, INDIGO, GREY, GREY], text=[pct(v) for v in detail.values], textposition="outside"))
fig.update_xaxes(range=[0, 1.12], tickformat=".0%")
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="2. Assignment group: how well are specialist teams and ITSC found, and how good is auto-routing?")
show(fig, height=360)

mistakes = pd.DataFrame({"actual": y, "predicted": st_out["assignment_group"]})
mistakes = mistakes[mistakes["actual"] != mistakes["predicted"]]
top = mistakes.groupby(["actual", "predicted"]).size().sort_values(ascending=False).head(10)
fig = go.Figure(go.Bar(y=[f"{a}  ->  {p}" for a, p in top.index], x=top.values, orientation="h", marker_color=RED, text=top.values, textposition="outside"))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="tickets")
fig.update_layout(title="The 10 most common assignment-group mistakes (actual -> predicted)", margin=dict(l=380))
show(fig, height=460)

# 3. confidence: coverage is the share answered at that confidence or higher, accuracy is measured on those only
conf_rows = []
for f, c in [("cmdb_ci", "cmdb_ci"), ("assignment_group", "assignment_group"), ("subcategory", "subcategory"), ("priority", "priority_label")]:
    for t in (0.0, 0.5, 0.7, 0.9):
        keep = st_out[f"{f}_conf"] >= t
        conf_rows.append({"field": f, "min_confidence": t, "coverage": keep.mean(), "accuracy_on_covered": (st_out[f][keep] == split_test[c][keep]).mean()})
conf = pd.DataFrame(conf_rows)
fig = make_subplots(rows=2, cols=2, subplot_titles=list(conf["field"].unique()), shared_yaxes=True)
for k, f in enumerate(conf["field"].unique()):
    d = conf[conf["field"] == f]
    r, col = k // 2 + 1, k % 2 + 1
    fig.add_scatter(x=d["min_confidence"], y=d["accuracy_on_covered"], mode="lines+markers+text", text=[f"{v:.0%}" for v in d["accuracy_on_covered"]], textposition="top center",
                    name="accuracy of the tickets answered", line=dict(color=INDIGO), showlegend=k == 0, row=r, col=col)
    fig.add_scatter(x=d["min_confidence"], y=d["coverage"], mode="lines+markers+text", text=[f"{v:.0%}" for v in d["coverage"]], textposition="bottom center",
                    name="coverage (share of tickets answered)", line=dict(color=GREY, dash="dash"), showlegend=k == 0, row=r, col=col)
fig.update_yaxes(range=[0, 1.1], tickformat=".0%")
fig.update_xaxes(title_text="minimum confidence", tickvals=[0, 0.5, 0.7, 0.9])
fig.update_layout(title="3. Confidence: the more confident, the more accurate, and the fewer tickets are answered")
show(fig, height=640)

# 4. urgent tickets (P1/P2), using the impact exactly as submitted
real_urgent = np.isin(split_test["priority_label"], URGENT)
flag = st_out["urgent_alert"].to_numpy()
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}] * 3])
fig.add_trace(go.Indicator(mode="number", value=flag[real_urgent].mean() * 100, number=dict(suffix="%", valueformat=".1f"), title=dict(text="urgent tickets caught by the alert")), row=1, col=1)
fig.add_trace(go.Indicator(mode="number", value=(real_urgent[flag].mean() * 100) if flag.any() else 0, number=dict(suffix="%", valueformat=".1f"), title=dict(text="alerts that were really urgent")), row=1, col=2)
fig.add_trace(go.Indicator(mode="number", value=flag.sum() / max(real_urgent.sum(), 1), number=dict(valueformat=".1f"), title=dict(text="alerts per real urgent ticket")), row=1, col=3)
fig.update_layout(title=f"4. Urgent (P1/P2) tickets: {int(real_urgent.sum())} of {len(split_test)} test tickets are urgent")
show(fig, height=300)

# 5. every test ticket: what it really was and what the model said (browse split_results, or save it with the line below)
split_results = pd.DataFrame({"number": split_test["number"], "short_description": split_test["short_description"],
                              "actual_ci": split_test["cmdb_ci"], "pred_ci": st_out["cmdb_ci"], "ci_ok": st_out["cmdb_ci"] == split_test["cmdb_ci"],
                              "actual_group": split_test["assignment_group"], "pred_group": st_out["assignment_group"], "group_conf": st_out["assignment_group_conf"].round(2),
                              "group_ok": st_out["assignment_group"] == split_test["assignment_group"],
                              "actual_priority": split_test["priority_label"], "pred_priority": st_out["priority"], "priority_ok": st_out["priority"] == split_test["priority_label"]})
# split_results.to_csv("split_results.csv", index=False)      # uncomment to save every test ticket to a file
wrong = split_results[~split_results["group_ok"]].head(15).copy()
wrong["short_description"] = wrong["short_description"].astype(str).str.replace(chr(10), " ").str.slice(0, 70)
table_figure(wrong[["number", "short_description", "actual_group", "pred_group", "group_conf", "actual_priority", "pred_priority"]].set_index("number"),
             f"5. split_results has {len(split_results)} rows: {int((~split_results['group_ok']).sum())} wrong assignment group, {int((~split_results['ci_ok']).sum())} wrong CI, "
             f"{int((~split_results['priority_ok']).sum())} wrong priority. A sample of tickets with a wrong assignment group:", col_widths=[1.2, 4, 3, 3, 1, 1.4, 1.4])

## 7. Confusion matrix

Rows are the actual value, columns are what the ensemble predicted; the diagonal is correct and `recall` shows how many of each actual class were found. Fields with many classes show the 10 biggest classes, the rest are grouped as `(other)`. Needs section 6 to have run first.

In [ ]:
def confusion(field, truth_col, top=None):
    # Rows = actual, columns = predicted. top=N keeps the N biggest actual classes and groups the rest as "(other)".
    actual, predicted = split_test[truth_col].astype(str), st_out[field].astype(str)
    if top:
        keep = actual.value_counts().head(top).index
        actual, predicted = actual.where(actual.isin(keep), "(other)"), predicted.where(predicted.isin(keep), "(other)")
    m = pd.crosstab(actual.rename("actual"), predicted.rename("predicted"))
    labels = sorted(set(m.index) | set(m.columns))
    return m.reindex(index=labels, columns=labels, fill_value=0)


for field, truth_col, top in [("priority", "priority_label", None), ("category", "category", None), ("cmdb_ci", "cmdb_ci", None),
                              ("assignment_group", "assignment_group", 10), ("subcategory", "subcategory", 10)]:
    m = confusion(field, truth_col, top)
    totals = m.sum(axis=1)
    diagonal = pd.Series({lab: m.loc[lab, lab] for lab in m.index})
    row_share = m.div(totals.replace(0, np.nan), axis=0)
    ylabels = [f"{lab}  (n={totals[lab]}, recall {diagonal[lab] / totals[lab]:.0%})" if totals[lab] else f"{lab}  (n=0)" for lab in m.index]
    share_text = [[f"{v:.0%}" if v == v and v > 0 else "" for v in row] for row in row_share.values]
    count_text = [[str(v) if v > 0 else "" for v in row] for row in m.values]
    fig = go.Figure()
    fig.add_heatmap(z=row_share.values, x=list(m.columns), y=ylabels, text=share_text, texttemplate="%{text}", colorscale="Blues", zmin=0, zmax=1, name="share",
                    colorbar=dict(title="share of the actual class", tickformat=".0%"), hovertemplate="actual: %{y}<br>predicted: %{x}<br>%{z:.1%} of the actual class<extra></extra>")
    fig.add_heatmap(z=m.values, x=list(m.columns), y=ylabels, text=count_text, texttemplate="%{text}", colorscale="Blues", name="tickets", visible=False,
                    colorbar=dict(title="tickets"), hovertemplate="actual: %{y}<br>predicted: %{x}<br>%{z} tickets<extra></extra>")
    fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.14, xanchor="left", buttons=[
        dict(label="share of each actual class", method="update", args=[{"visible": [True, False]}]),
        dict(label="ticket counts", method="update", args=[{"visible": [False, True]}])])],
        title=f"{field}: actual (rows) against predicted (columns)" + (f", the {top} biggest classes" if top else "") + "<br><sup>the diagonal is correct; recall of each actual class is in its row label</sup>",
        xaxis_title="predicted", yaxis_title="actual", margin=dict(l=330 if top else 230))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(tickangle=-40, side="bottom")
    show(fig, height=max(460, 160 + 50 * len(m)))

## 8. Test your own new tickets

Needs sections 0 to 4. It uses the ensemble and does not train anything. Fill in `my_tickets`; the `expected_...` lines are the answers **you know are correct** (all optional). For each ticket it prints the ensemble's predictions and marks them OK or WRONG, then gives an accuracy per field over the tickets where you gave an answer. Tickets that were not in the CSV give an honest score.

In [ ]:
model = ens_model   # the ensemble from section 4. For the ensemble trained on ALL tickets (section 9) use: model = joblib.load(FULL_PATH)

# One dict per ticket. Copy the first one and change it. "cmdb_ci" is what the requester selected (may be wrong; leave "" if unknown).
my_tickets = [
    {
        "short_description": "Unable to open Outlook, mailbox full",
        "description": "Cannot send or receive email since this morning",
        "contact_type": "Phone",
        "opened_at": "15-09-2026 10:15:00",
        "impact": "3 - Low",
        "cmdb_ci": "",
        "expected_cmdb_ci": "S.A.MS-OUTLOOK",              # the answers you know are right (all optional)
        "expected_assignment_group": "IMG-SMC-TCC-ITSC",
        "expected_category": "Software",
        "expected_subcategory": "Email Related",
        "expected_priority": "P3",
    },
    # {
    #     "short_description": "...",
    #     "description": "...",
    #     "contact_type": "Phone",
    #     "opened_at": "dd-mm-yyyy hh:mm:ss",
    #     "impact": "2 - Medium",
    #     "cmdb_ci": "",
    #     "expected_cmdb_ci": "",
    #     "expected_assignment_group": "",
    #     "expected_category": "",
    #     "expected_subcategory": "",
    #     "expected_priority": "",
    # },
]

OK_FILL, WRONG_FILL, PLAIN_FILL = "#dcfce7", "#fee2e2", "white"
hits = {f: [] for f in FIELDS}
for n, t in enumerate(my_tickets, 1):
    r = ensemble_predict_ticket({k: v for k, v in t.items() if not k.startswith("expected_")}, model)
    rows, fills, bar_colors = [], [], []
    for f in FIELDS:
        want = t.get(f"expected_{f}")
        verdict = "-"
        fill = PLAIN_FILL
        if want:
            ok = r[f] == want
            hits[f].append(ok)
            verdict, fill = ("OK", OK_FILL) if ok else (f"WRONG (you expected {want})", WRONG_FILL)
        rows.append((f, r[f], f"{r[f + '_conf']:.0%}", verdict))
        fills.append(fill)
        bar_colors.append(GREEN if fill == OK_FILL else RED if fill == WRONG_FILL else INDIGO)
    rows += [("auto-route", "yes" if r["auto_route"] else "no, send to a human / ITSC", "", ""),
             ("urgent alert", ("YES (" + r["alert_reason"] + ")") if r["urgent_alert"] else "no", "", ""),
             ("CI mismatch", ("YES, the text suggests " + r["cmdb_ci"]) if r["ci_mismatch"] else "no", "", "")]
    fills += [PLAIN_FILL] * 3
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], specs=[[{"type": "table"}, {"type": "xy"}]])
    fig.add_trace(go.Table(columnwidth=[1.4, 3, 1, 2.6], header=dict(values=["<b>field</b>", "<b>predicted</b>", "<b>confidence</b>", "<b>your answer</b>"], fill_color="#e0e7ff", align="left"),
                           cells=dict(values=[[x[i] for x in rows] for i in range(4)], align="left", height=28, fill_color=[[f for f in fills]] * 4)), row=1, col=1)
    fig.add_trace(go.Bar(y=FIELDS, x=[r[f"{f}_conf"] for f in FIELDS], orientation="h", marker_color=bar_colors, text=[f"{r[f + '_conf']:.0%}" for f in FIELDS], textposition="outside",
                         showlegend=False), row=1, col=2)
    fig.update_xaxes(range=[0, 1.15], tickformat=".0%", title_text="confidence", row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_layout(title=f"Ticket {n}: {t['short_description']}<br><sup>models: " + ", ".join(f"{f} = {MODEL_NAME[model['fields'][f]['chosen']]}" for f in FIELDS) + "</sup>")
    show(fig, height=440)

done = [(f, sum(hits[f]), len(hits[f])) for f in FIELDS if hits[f]]
if done:
    table_figure(pd.DataFrame([(f, f"{a} of {b} correct", f"{a / b:.0%}") for f, a, b in done], columns=["field", "correct", "accuracy"]).set_index("field"),
                 "Accuracy on the tickets where you gave an answer")
else:
    print("(no expected_... answers given, so only the predictions above are shown)")

## 9. (Optional) Train the ensemble on all tickets and save it

For real predictions: the same ensemble, with the same settings, trained on **all** tickets and saved as `triage_ensemble_model_full.joblib`. It takes about as long as section 4 again. It cannot be scored honestly on this data (it has seen every ticket), which is why sections 5 to 7 use the 80% ensemble. The model chosen for each field is chosen again on all tickets, so it can differ from section 4.

Set `TRAIN_ON_ALL = True` to run it; the default `False` does nothing.

In [ ]:
TRAIN_ON_ALL = False   # set to True to train the ensemble on all tickets and save it
FULL_PATH = Path(f"triage_ensemble_model_full{TAG}.joblib")

if TRAIN_ON_ALL:
    print(f"training the ensemble on all {len(all_tickets)} tickets ...")
    full_model = fit_ensemble(all_tickets, BASE_MODELS, ENSEMBLE_FOLDS)
    full_model["settings"]["trained_on"] = "all tickets"
    joblib.dump(full_model, FULL_PATH, compress=3)
    print(f"saved {FULL_PATH.name} ({FULL_PATH.stat().st_size / 1e6:.0f} MB)")
else:
    print("TRAIN_ON_ALL is False: nothing was trained or saved.")


## 10. Reference: the CIs the model is trained on, and the columns it needs

Run after section 4. The CI list is read from the saved hybrid (`triage_ensemble_model.joblib`, earliest 80% of tickets), so it shows exactly what the model can predict. A CI that is not in the list can never be predicted. Every CI that appears in training is kept.

The columns table says what to put in `open_incidents.csv` (or any new tickets) before calling `ensemble_predict_df()`. Missing columns do not stop the model: `prepare()` turns them into `unknown`, but the fields that use them get less accurate. `cmdb_ci` is the text-only model, so it is not affected by the missing context columns.

In [ ]:
assert "ens_model" in globals(), "Run section 4 first"
trained_ci = list(ens_model["fields"]["cmdb_ci"]["classes"])
ci_reference = pd.DataFrame({
    "tickets in training": split_train["cmdb_ci"].value_counts().reindex(trained_ci),
    "tickets in test": split_test["cmdb_ci"].value_counts().reindex(trained_ci).fillna(0).astype(int),
}).sort_values("tickets in training", ascending=False)
ci_reference["note"] = ["" if n >= 1000 else "weak: few training tickets" if n >= 50 else "very weak: too few training tickets" for n in ci_reference["tickets in training"]]
table_figure(ci_reference, f"The ensemble predicts {len(trained_ci)} CIs (trained on the earliest {len(split_train)} tickets). A CI that is not in this list can never be predicted.",
             col_widths=[3, 1.4, 1.2, 3])

columns_needed = pd.DataFrame([
    ("short_description", "text for every model",                          "yes",      "-"),
    ("description",       "text for every model",                          "yes",      "-"),
    ("opened_at",         "hour + weekday, used by assignment_group",      "optional", "-1 (unknown); format dd-mm-yyyy hh:mm:ss"),
    ("contact_type",      "assignment_group",                              "optional", "unknown"),
    ("caller_id",         "assignment_group (caller name as exported)",    "optional", "unknown"),
    ("sys_created_by",    "assignment_group",                              "optional", "unknown"),
    ("impact",            "priority",                                      "optional", "unknown"),
    ("cmdb_ci",           "the CI the requester picked: only for ci_mismatch, never a feature", "optional", "no mismatch check"),
    ("number",            "ticket id, only to identify rows",              "optional", "-"),
], columns=["column", "used for", "needed", "if missing"]).set_index("column")
needed_fill = ["#fde68a" if v == "yes" else "white" for v in columns_needed["needed"]]
table_figure(columns_needed, f"COLUMNS for predicting new tickets (INPUT_COLUMNS = {INPUT_COLUMNS}). Only for training (the raw_incident_reassign.csv export): number, assignment_group, category, "
             "subcategory, priority, plus the columns above.", col_widths=[1.6, 4, 1, 3], fill_colors=[["white"] * len(columns_needed), ["white"] * len(columns_needed),
                                                                                                    needed_fill, ["white"] * len(columns_needed), ["white"] * len(columns_needed)])